# 🎬 ShimiStudio — Cloud Worker v4.3 (רוטציית ענן חינם)
### Kaggle — GPU P100/T4x2, 30 שעות GPU בשבוע (ה-worker נרשם ל-8 שעות ואז מתנתק בסבב)

**מה זה עושה:** המחברת מתקינה את ComfyUI על ה-GPU החינמי, מורידה את המודלים, ומריצה את worker v4.3 — **אותו worker בדיוק כמו במחשב הביתי** — שנרשם לתור הרינדור של ShimiStudio ומחכה לעבודות. העבודות שנשלחות בסטודיו במצב **"ענן"** יגיעו לכאן.

**הוראות:**
1. ודא ש-GPU דלוק (Settings → Accelerator → GPU P100 / T4 x2 + Internet = ON)
2. הרץ את כל התאים לפי הסדר
3. זהו. אין יותר "העתק URL והדבק בסטודיו" — ה-worker מתחבר לתור לבד
4. כשהסשן נגמר — פשוט מפעילים את המחברת מחדש (רוטציה חינם)

⚠️ כשהמחברת רצה — אל תסגור את הדפדפן לזמן ארוך, וודא שהתא האחרון ממשיך לרוץ.

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name} | 💾 VRAM: {vram:.1f} GB')
    assert vram >= 10, '❌ צריך לפחות 10GB VRAM'
else:
    raise RuntimeError('❌ אין GPU! הפעל GPU לפי ההוראות למעלה והרץ מחדש')

In [ ]:
import os, subprocess
COMFY_PATH = '/kaggle/working/ComfyUI'
CUSTOM = COMFY_PATH + '/custom_nodes'
if not os.path.exists(COMFY_PATH):
    print('⬇️ מתקין ComfyUI...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY_PATH], check=True)
os.chdir(COMFY_PATH)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
nodes = {
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git',
    'ComfyUI-ReActor': 'https://github.com/Gourieff/ComfyUI-ReActor.git',
    'ComfyUI-IPAdapter-plus': 'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git',
}
for nm, url in nodes.items():
    p = f'{CUSTOM}/{nm}'
    if not os.path.exists(p):
        print(f'⬇️ {nm}')
        subprocess.run(['git', 'clone', '--depth', '1', url, p], check=True)
subprocess.run(['pip', 'install', '-q', 'imageio-ffmpeg', 'imageio[ffmpeg]'], check=True)
print('✅ ComfyUI + custom nodes מוכנים')

In [ ]:
import os, requests
M = '/kaggle/working/ComfyUI/models'
paths = {
    M + '/checkpoints': 'v1-5-pruned-emaonly.safetensors|https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
    M + '/ipadapter': 'ip-adapter-plus_sd15.safetensors|https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors',
    M + '/clip_vision': 'CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors|https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
    M + '/animatediff_models': 'mm_sd_v15_v2.ckpt|https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt',
}
for d, spec in paths.items():
    os.makedirs(d, exist_ok=True)
    fname, url = spec.split('|')
    dest = os.path.join(d, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        print(f'✅ {fname} כבר קיים')
        continue
    print(f'⬇️ מוריד {fname}...')
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    done = 0
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            done += len(chunk)
            if total and (done*100//total) % 10 == 0:
                print(f'   {done*100//total}% ({done//1048576}/{total//1048576} MB)', flush=True)
print('✅ כל המודלים מוכנים')

In [ ]:
import os, re, json, secrets, sys, requests
from datetime import datetime, timezone, timedelta
WORKER_DIR = '/kaggle/working/ShimiWorker'
os.makedirs(WORKER_DIR, exist_ok=True)
TOKEN = secrets.token_hex(12)
NAME = 'Worker-Kaggle-T4'
API_BASE = 'https://shimi-studio.base44.app/api/apps/6a9206e9f29b8d9f70a77b47'
SESSION_HOURS = 8
src = requests.get('https://raw.githubusercontent.com/sunraz/ShimiStudio/main/worker_v43.py', timeout=60).text
assert 'workerApi' in src, 'קובץ ה-worker לא ירד תקין'
src = re.sub(r'VENV_PY = .*', 'VENV_PY = sys.executable', src, count=1)
exp = (datetime.now(timezone.utc) + timedelta(hours=SESSION_HOURS)).isoformat()
src = src.replace('"action": "register",', '"action": "register", "worker_type": "cloud", "session_expires_at": "' + exp + '",', 1)
with open(WORKER_DIR + '/worker.py', 'w') as f:
    f.write(src)
with open(WORKER_DIR + '/config.json', 'w') as f:
    json.dump({"server": API_BASE.rsplit('/api/apps', 1)[0], "token": TOKEN, "name": NAME,
                "apiBase": API_BASE, "comfyui_path": '/kaggle/working/ComfyUI',
                "comfyui_url": "http://127.0.0.1:8188"}, f)
print(f'✅ Worker v4.3 (cloud) מוכן | שם: {NAME} | תוקף סשן: {SESSION_HOURS} שעות')
print(f'   Token: {TOKEN[:8]}...')

In [ ]:
import os, sys
os.chdir('/kaggle/working/ShimiWorker')
print('🚀 מעלה את ComfyUI ומתחבר לתור של ShimiStudio...')
print('   השאר את התא הזה רץ — כל עוד הוא רץ, ה-GPU הזה משרת את הסטודיו')
print()
raise SystemExit(os.system(sys.executable + ' worker.py') >> 8)